# Korelasi Lag Spasial Curah Hujan vs Nino-3.4

Notebook ini menghitung korelasi lag spasial untuk curah hujan DJF terhadap Nino-3.4 rata-rata berjalan 3 bulan yang terpusat, lalu menyimpan cache NetCDF agar plot dapat diubah tanpa menghitung ulang.

Yang dihitung:
- 1981-2025
- P1 = 1981-2006
- P2 = 2007-2025
- selisih P2-P1

Notebook ini memakai gaya peta yang sama seperti `plot_correlation_mc.ipynb`.


## Konvensi Lag

- `lag 0` = `DJF`
- `lag -1` = `NDJ`
- `lag -2` = `OND`
- `lag -3` = `SON`
- `lag 1` = `JFM`
- `lag 2` = `FMA`
- `lag 3` = `MAM`

Untuk notebook ini, `lag` dipakai sebagai pergeseran bulan pada seri Nino-3.4 yang sudah dirata-ratakan 3 bulan.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from scipy.ndimage import gaussian_filter
from scipy.stats import norm, t as student_t

plt.rcParams.update({
    'font.family': 'Helvetica',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
})

PROJECT_ROOT = Path('../../..').resolve()
DATA_ROOT = Path('../../external/ClimateData').resolve()
CACHE_DIR = Path('../../data/intermediate/divided_correlation').resolve()
RESULTS_DIR = Path('../../results/analisis_korelasi_26-19').resolve()
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RAINFALL_PATH = Path('../../external/ClimateData/mswep-monthly/mswep_monthly_combined.nc').resolve()
NINO34_PATH = Path('../../external/ClimateData/index-monthly/nino34.anom.csv').resolve()
OUT_NC = CACHE_DIR / 'rainfall_spatial_lagcorr_1981_2025.nc'

START_YEAR = 1981
END_YEAR = 2025
P1 = (1981, 2006)
P2 = (2007, 2025)
LAGS = np.arange(-12, 13)
PLOT_LAG = 0

MC_EXTENT = [80, 160, -20, 20]
MC_XTICKS = np.arange(90, 161, 20)
MC_YTICKS = np.arange(-20, 21, 10)
CORR_LEVELS = np.arange(-1, 1.01, 0.05)
CORR_TICKS = np.arange(-1, 1.01, 0.25)

LAG_TITLES = {
    -12: '-DJF',
    -11: '-JFM',
    -10: '-FMA',
    -9: '-MAM',
    -8: '-AMJ',
    -7: '-MJJ',
    -6: '-JJA',
    -5: '-JAS',
    -4: '-ASO',
    -3: '-SON',
    -2: '-OND',
    -1: '-NDJ',
    0: 'DJF',
    1: 'JFM',
    2: 'FMA',
    3: 'MAM',
    4: 'AMJ',
    5: 'MJJ',
    6: 'JJA',
    7: 'JAS',
    8: 'ASO',
    9: 'SON',
    10: 'OND',
    11: 'NDJ',
    12: 'DJF+',
}


def lag_label(lag: int) -> str:
    return LAG_TITLES.get(int(lag), f'{int(lag):+d}')


def save_plot(fig, filename):
    fig.savefig(RESULTS_DIR / filename, dpi=300, bbox_inches='tight')


def smooth_for_plot(da, sigma=0.7):
    if not {'lat', 'lon'}.issubset(da.dims):
        return da
    def _smooth(arr):
        return gaussian_filter(arr, sigma=sigma)
    smoothed = xr.apply_ufunc(
        _smooth,
        da,
        input_core_dims=[['lat', 'lon']],
        output_core_dims=[['lat', 'lon']],
        vectorize=True,
        dask='allowed',
        output_dtypes=[da.dtype],
    )
    smoothed = smoothed.transpose(*da.dims)
    smoothed = smoothed.assign_attrs(da.attrs)
    if da.name is not None:
        smoothed = smoothed.rename(da.name)
    return smoothed


def standardize_rainfall_da(da: xr.DataArray) -> xr.DataArray:
    rename_map = {}
    if 'latitude' in da.dims or 'latitude' in da.coords:
        rename_map['latitude'] = 'lat'
    if 'longitude' in da.dims or 'longitude' in da.coords:
        rename_map['longitude'] = 'lon'
    if 'valid_time' in da.dims or 'valid_time' in da.coords:
        rename_map['valid_time'] = 'time'
    if rename_map:
        da = da.rename(rename_map)
    da = da.assign_coords(time=pd.to_datetime(da['time'].values)).sortby('time')
    da = da.assign_coords(lon=(da['lon'] % 360)).sortby('lon')
    return da


def load_nino34_monthly(path: Path) -> pd.Series:
    df = pd.read_csv(path)
    if df.shape[1] < 2:
        raise ValueError(f'Nino3.4 file must have at least 2 columns, got {df.columns.tolist()}')
    date_col = None
    value_col = None
    for col in df.columns:
        low = str(col).lower()
        if date_col is None and low in {'date', 'time', 'month', 'ym'}:
            date_col = col
        elif value_col is None and low in {'nino34', 'nino3.4', 'nino-3.4', 'anom', 'anomaly', 'value'}:
            value_col = col
    if date_col is None:
        date_col = df.columns[0]
    if value_col is None:
        value_col = df.columns[1]
    s = pd.Series(pd.to_numeric(df[value_col], errors='coerce').to_numpy(), index=pd.to_datetime(df[date_col]))
    s = s.sort_index().groupby(s.index.to_period('M')).mean()
    s.index = s.index.to_timestamp(how='start')
    s.name = 'nino34'
    return s


def build_centered_3mo_running_mean(series: pd.Series) -> pd.Series:
    return series.rolling(3, center=True, min_periods=3).mean().dropna()


def build_djf_year_coord(time: xr.DataArray) -> xr.DataArray:
    return xr.where(time.dt.month == 12, time.dt.year + 1, time.dt.year)


def prep_rainfall_field(path: Path) -> xr.DataArray:
    ds = xr.open_dataset(path)
    rain_var = 'precipitation' if 'precipitation' in ds.data_vars else list(ds.data_vars)[0]
    rain = standardize_rainfall_da(ds[rain_var])
    rain = rain.sel(time=slice('1979-12-01', '2025-02-28'))
    rain = rain.sel(lat=slice(20, -20), lon=slice(80, 160))
    rain = rain.chunk({'time': 36, 'lat': 120, 'lon': 120}).astype(np.float32)
    rain_djf = rain.sel(time=rain.time.dt.month.isin([12, 1, 2]))
    rain_djf = rain_djf.assign_coords(djf_year=('time', build_djf_year_coord(rain_djf.time).data))
    return rain_djf


def build_nino_lag_series(nino_3mo: pd.Series, years: np.ndarray, lag: int) -> xr.DataArray:
    vals = []
    idx = []
    for year in years:
        ts = pd.Timestamp(int(year), 1, 1) + pd.DateOffset(months=int(lag))
        if ts in nino_3mo.index:
            vals.append(float(nino_3mo.loc[ts]))
            idx.append(int(year))
    return xr.DataArray(np.array(vals, dtype=float), coords={'djf_year': np.array(idx, dtype=int)}, dims='djf_year', name=f'nino34_lag_{lag:+d}')


def build_nino_lag_matrix(nino_3mo: pd.Series, years: np.ndarray, lags: np.ndarray) -> xr.DataArray:
    year_index = np.array(years, dtype=int)
    lag_series = []
    for lag in np.array(lags, dtype=int):
        lag_da = build_nino_lag_series(nino_3mo, year_index, int(lag)).reindex(djf_year=year_index)
        lag_series.append(lag_da.assign_coords(lag=int(lag)).expand_dims('lag'))
    lag_matrix = xr.concat(lag_series, dim='lag').transpose('lag', 'djf_year')
    return lag_matrix.astype(np.float32)


def corr_map_lags(da_djf: xr.DataArray, nino_lag_matrix: xr.DataArray):
    da_a, nino_a = xr.align(da_djf, nino_lag_matrix, join='inner')
    corr = xr.corr(da_a, nino_a, dim='djf_year').transpose('lag', 'lat', 'lon')
    n = nino_a.notnull().sum('djf_year').astype(np.int16)
    safe = xr.where(np.abs(corr) >= 0.999999, np.sign(corr) * 0.999999, corr)
    n_eff = n.broadcast_like(corr)
    t_stat = safe * np.sqrt((n_eff - 2) / (1 - safe ** 2))
    pval = xr.apply_ufunc(
        lambda t, df: 2 * student_t.sf(np.abs(t), df=df),
        t_stat,
        n_eff - 2,
        dask='parallelized',
        output_dtypes=[float],
    )
    pval = pval.where(n_eff > 2)
    return corr.astype(np.float32), pval.astype(np.float32), n


def fisher_diff_pvalue_map(r1, n1, r2, n2):
    r1 = xr.where(np.abs(r1) >= 0.999999, np.sign(r1) * 0.999999, r1)
    r2 = xr.where(np.abs(r2) >= 0.999999, np.sign(r2) * 0.999999, r2)
    n1_eff = n1.broadcast_like(r1)
    n2_eff = n2.broadcast_like(r2)
    z = (np.arctanh(r1) - np.arctanh(r2)) / np.sqrt(1 / (n1_eff - 3) + 1 / (n2_eff - 3))
    pval = xr.apply_ufunc(
        lambda x: 2 * norm.sf(np.abs(x)),
        z,
        dask='parallelized',
        output_dtypes=[float],
    )
    return pval.where((n1_eff > 3) & (n2_eff > 3)).astype(np.float32)


def plot_mc_map(data, sig_mask, title, filename, cbar_label):
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))
    plot_data = smooth_for_plot(data).reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=CORR_LEVELS,
        extend='both',
        add_colorbar=False,
        infer_intervals=True,
    )

    sig_plot = sig_mask.fillna(False).astype(np.int8)
    sig_plot = sig_plot.coarsen(lat=8, lon=8, boundary='trim').max() > 0
    yy, xx = np.where(sig_plot.values)
    if yy.size > 0:
        ax.scatter(
            sig_plot['lon'].values[xx],
            sig_plot['lat'].values[yy],
            s=15,
            c='k',
            marker='.',
            linewidths=0,
            alpha=0.7,
            transform=ccrs.PlateCarree(),
            zorder=4,
        )

    ax.set_extent(MC_EXTENT, crs=ccrs.PlateCarree())
    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(MC_XTICKS, crs=ccrs.PlateCarree())
    ax.set_yticks(MC_YTICKS, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=20)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=CORR_TICKS, spacing='proportional', extend='both')
    cbar.set_label(cbar_label, fontsize=13, labelpad=10)
    cbar.ax.tick_params(labelsize=12)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


In [ ]:
# Load data
rain_djf = prep_rainfall_field(RAINFALL_PATH)
nino_monthly = load_nino34_monthly(NINO34_PATH)
nino_3mo = build_centered_3mo_running_mean(nino_monthly)
nino_3mo = nino_3mo.loc['1980-01-01':'2026-01-01']

full_years = np.arange(START_YEAR, END_YEAR + 1)
past_years = np.arange(P1[0], P1[1] + 1)
recent_years = np.arange(P2[0], P2[1] + 1)

rain_djf_annual = rain_djf.groupby('djf_year').mean('time').chunk({'djf_year': -1, 'lat': 120, 'lon': 120})

rain_clim = rain_djf_annual.where(rain_djf_annual.coords['djf_year'].isin(full_years), drop=True)
rain_past = rain_clim.where(rain_clim.coords['djf_year'].isin(past_years), drop=True)
rain_recent = rain_clim.where(rain_clim.coords['djf_year'].isin(recent_years), drop=True)

print('Rainfall path:', RAINFALL_PATH)
print('Nino3.4 path:', NINO34_PATH)
print('Rainfall DJF shape:', rain_clim.shape)


In [ ]:
# Compute lagged spatial correlations and save to cache
nino_lag_full = build_nino_lag_matrix(nino_3mo, full_years, LAGS)
nino_lag_past = nino_lag_full.sel(djf_year=past_years)
nino_lag_recent = nino_lag_full.sel(djf_year=recent_years)

corr_clim, p_clim, n_clim = corr_map_lags(rain_clim, nino_lag_full)
corr_past, p_past, n_past = corr_map_lags(rain_past, nino_lag_past)
corr_recent, p_recent, n_recent = corr_map_lags(rain_recent, nino_lag_recent)
p_diff = fisher_diff_pvalue_map(corr_recent, n_recent, corr_past, n_past)

spatial_lagcorr = xr.Dataset(
    {
        'rain_corr_clim': corr_clim,
        'rain_corr_past': corr_past,
        'rain_corr_recent': corr_recent,
        'rain_corr_recent_minus_past': (corr_recent - corr_past).astype(np.float32),
        'rain_corr_clim_sig': p_clim,
        'rain_corr_past_sig': p_past,
        'rain_corr_recent_sig': p_recent,
        'rain_corr_recent_minus_past_sig': p_diff,
    }
)

encoding = {
    var_name: {'zlib': True, 'complevel': 3, 'dtype': 'float32'}
    for var_name in spatial_lagcorr.data_vars
}
spatial_lagcorr.to_netcdf(OUT_NC, encoding=encoding)
print('Saved cache:', OUT_NC)
print(spatial_lagcorr)


## Plot Lag Terpilih

Notebook ini hanya mem-plot satu lag yang dipilih lewat `PLOT_LAG`. Kalau ingin melihat lag lain, ubah nilainya lalu jalankan ulang sel plot saja.


In [ ]:
# Plot the selected lag using the same map style as plot_correlation_mc.ipynb
plot_lag_label = lag_label(PLOT_LAG)
plot_lag_index = int(np.where(LAGS == PLOT_LAG)[0][0])

rain_plot_clim = spatial_lagcorr['rain_corr_clim'].isel(lag=plot_lag_index)
rain_plot_past = spatial_lagcorr['rain_corr_past'].isel(lag=plot_lag_index)
rain_plot_recent = spatial_lagcorr['rain_corr_recent'].isel(lag=plot_lag_index)
rain_plot_diff = spatial_lagcorr['rain_corr_recent_minus_past'].isel(lag=plot_lag_index)

rain_plot_clim_sig = spatial_lagcorr['rain_corr_clim_sig'].isel(lag=plot_lag_index)
rain_plot_past_sig = spatial_lagcorr['rain_corr_past_sig'].isel(lag=plot_lag_index)
rain_plot_recent_sig = spatial_lagcorr['rain_corr_recent_sig'].isel(lag=plot_lag_index)
rain_plot_diff_sig = spatial_lagcorr['rain_corr_recent_minus_past_sig'].isel(lag=plot_lag_index)

plot_mc_map(
    rain_plot_clim,
    rain_plot_clim_sig,
    f'Korelasi Curah Hujan vs Nino34 DJF 1981-2025 ({plot_lag_label})',
    f'rainfall_spatial_corr_{P1[0]}-{P2[1]}_lag_{PLOT_LAG:+d}.png',
    'Koefisien korelasi curah hujan',
)

plot_mc_map(
    rain_plot_past,
    rain_plot_past_sig,
    f'Korelasi Curah Hujan vs Nino34 DJF P1 ({plot_lag_label})',
    f'rainfall_spatial_corr_p1_lag_{PLOT_LAG:+d}.png',
    'Koefisien korelasi curah hujan',
)

plot_mc_map(
    rain_plot_recent,
    rain_plot_recent_sig,
    f'Korelasi Curah Hujan vs Nino34 DJF P2 ({plot_lag_label})',
    f'rainfall_spatial_corr_p2_lag_{PLOT_LAG:+d}.png',
    'Koefisien korelasi curah hujan',
)

plot_mc_map(
    rain_plot_diff,
    rain_plot_diff_sig,
    f'Selisih Korelasi Curah Hujan P2-P1 ({plot_lag_label})',
    f'rainfall_spatial_corr_p2_minus_p1_lag_{PLOT_LAG:+d}.png',
    'Selisih koefisien korelasi curah hujan',
)
